# MFA-Modell für Roggenstroh - Modulare Version

## Changelog / Änderungsprotokoll

* **Version:** 1.0
* **Datum:** 14.06.2025
* **Autor:** [JS]
* **Änderungen:**
    * Grundstruktur für modulares Notebook erstellt.
    * Code in Funktionsblöcke unterteilt (Setup, Berechnungen, Visualisierung).

# Section 0: Importing the packages & basic settings translator

In [1]:
### Load packages ###

# Load general libraries
import sys, os
import numpy as np
import pandas as pd
from scipy.stats import lognorm
import xlsxwriter
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator,
                               FormatStrFormatter,
                               AutoMinorLocator)
import warnings
import re
from collections import defaultdict
from scipy.optimize import minimize
import copy

# Load ODYM package
# Add ODYM module directory to system path, absolute
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())),'framework', 'ODYM-master_20241127', 'odym', 'modules')) 

# Import the ODYM class file
import ODYM_Classes as msc 
# Import the ODYM function file
import ODYM_Functions as msf
# Import the dynamic stock model library
import dynamic_stock_model as dsm 


# Load bioDYM_addon
# Add ODYM module directory to system path, absolute
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'framework', 'bioDYM_add-on', 'modules')) 

# Import classes for first order model process
import bioDYM_classes as bicl
# Import plotting functions
import bioDYM_plotting as bipl
# Import export functions
import bioDYM_export as bix




# Enables plotting directly in the notebook (in most cases this is already enabled)
%matplotlib inline

# Section 1: Configuration

In [2]:
# ====================================================================
# Section 1: Model Configuration
# All user-defined settings for a scenario run go here.
# ====================================================================

# --- File Paths ---
EXCEL_FILE_PATH = '250617_Template_CS0_3.xlsx'

# --- Model Scope ---
START_YEAR = 2025
END_YEAR = 2050
ELEMENTS = ['material', 'WC', 'DM', 'CC'] # Elements to be tracked

# In Section 1: Model Configuration

# --- Model Calculation Switches ---
# Set to True to enable the calculation, False to disable.
# This allows for simpler test runs without DSM or FOMP.
RUN_DSM_CALCULATION = True
RUN_FOMP_CALCULATION = True

# --- Dynamic Stock Model (DSM) Parameters ---
# This dictionary holds the parameters for ALL dynamic stocks in the model.
# The key (e.g., 3) is the integer process_id where the stock is located.
# This structure allows for multiple DSMs; just add another process_id as a key.
# In Section 1: Model Configuration
DSM_PARAMS = {
    6: { # Parameter für das Lager im Prozess 3 (MBC_Use-Phase)
        'inflow_split': [0.8, 0.1, 0.1], # 20%, 20%, 60% Aufteilung
        'lifetimes': {
            'Type': 'Normal',
            'Mean': [20, 10, 5],
            'StdDev': [1.0, 0.5, 0.2]
        },
        # NEU: Namen für die Produktkategorien in der Legende der Grafik
        'category_names': ['Insulation Material', 'Panel Board', 'Packaging'] 
    }
}
    # Example for a second dynamic stock in another process (e.g., process 25):
    # 25: {
    #     'lifetimes': {
    #         'Type': 'Lognormal',
    #         'Mean': [50],
    #         'StdDev': [15]
    #     }
    # }

# --- First-Order Model Process (FOMP) Parameters ---
# This dictionary holds parameters for ALL FOMP calculations.
# The key (e.g., 17) is the integer process_id where the decay occurs.
# In Section 1: Model Configuration

FOMP_PARAMS = {
    8: { # Parameter für die Mineralisierung im Prozess 17 (Lithosphere_Stock)
        'outflow_id': 'F_08_00', # NEU: Der exakte Name des Abflusses, der berechnet werden soll
        'f': 0.236, 
        'k1': 0.25, 
        'k2': 0.0351
    }
}
    
    # Example for a second FOMP in another process (e.g., process 16):
    # 16: {
    #     'f': 0.1,
    #     'k1': 0.05,
    #     'k2': 0.001
    # }

# Section 2: Function definitions

In [3]:
# ====================================================================
# Section 2: Function Definitions
# ====================================================================

# --- Helper Function 2.1: Define Model Scope & Classifications ---
def define_model_scope(start_year, end_year, elements):
    """
    Defines the temporal and elemental scope of the MFA model.

    Args:
        start_year (int): The first year of the analysis.
        end_year (int): The last year of the analysis.
        elements (list): A list of strings for the elements to be tracked.

    Returns:
        tuple: A tuple containing the ModelClassification dictionary 
               and the IndexTable DataFrame, which are core ODYM objects.
    """
    ModelClassification = {}
    MyYears = list(np.arange(start_year, end_year + 1))
    
    # Define Time and Element Classifications for the ODYM framework
    ModelClassification['Time'] = msc.Classification(Name='Time', Dimension='Time', ID=1, Items=MyYears)
    ModelClassification['Element'] = msc.Classification(Name='Elements', Dimension='Element', ID=2, Items=elements)

    # Create the IndexTable, which ODYM uses for calculations
    IndexTable = pd.DataFrame({
        'Aspect': ['Time', 'Element'],
        'Description': ['Model aspect "time"', 'Model aspect "Element"'],
        'Dimension': ['Time', 'Element'],
        'Classification': [ModelClassification[Aspect] for Aspect in ['Time', 'Element']],
        'IndexLetter': ['t', 'e']
    })
    IndexTable.set_index('Aspect', inplace=True)
    
    print("--> Model scope and classifications defined.")
    return ModelClassification, IndexTable

In [4]:
# --- Helper Function 2.2: Initialize the main MFA System object ---
def initialize_mfa_system(model_classification, index_table):
    """
    Initializes the main MFAsystem object based on the defined scope.

    Args:
        model_classification (dict): The ModelClassification dictionary from define_model_scope.
        index_table (pd.DataFrame): The IndexTable DataFrame from define_model_scope.

    Returns:
        odym.MFAsystem: An empty but structured MFAsystem object.
    """
    # Extract scope details from the input objects
    start_time = model_classification['Time'].Items[0]
    end_time = model_classification['Time'].Items[-1]
    element_items = model_classification['Element'].Items
    
    # Create the main system object using the ODYM class
    MFA_System = msc.MFAsystem(
        Name='RyeStrawMFA', 
        Geogr_Scope='Case_Study_Region', 
        Unit='Mg', 
        ProcessList=[], 
        FlowDict={}, 
        StockDict={},
        ParameterDict={}, 
        Time_Start=start_time, 
        Time_End=end_time, 
        IndexTable=index_table, 
        Elements=element_items
    )
    
    print("--> MFA system object initialized.")
    return MFA_System

In [5]:
# Helper Function 2.3: Load data and define processes and stocks (Updated)
def load_and_define_processes(mfa_system, excel_path):
    """
    Loads all data from the Excel file, treating 'N.A.' as a missing value,
    and populates the ProcessList and StockDict in the MFA system.
    """
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
    
    # Load all sheets, now recognizing "N.A." as a blank cell (NaN)
    input_data = pd.read_excel(
        excel_path, 
        sheet_name=None, 
        header=0, 
        engine='openpyxl',
        na_values=['N.A.', 'NA', 'n/a'] # NEW: This tells pandas to ignore these values
    )
    print(f"--> Excel file '{excel_path}' loaded successfully.")
    
    process_definitions = input_data['2_1_Definition_Processes']
    
    for index, row in process_definitions.iterrows():
        # The 'pd.notna' check now also correctly handles the "N.A." strings
        if pd.notna(row['Name(EN)']):
            process_id = int(row['ID'])
            has_tcs = 'TC' if row['TC?'] == 'Yes' else 'None'
            
            mfa_system.ProcessList.append(
                msc.Process(Name=row['Name(EN)'], ID=process_id, Extensions=has_tcs)
            )
            
            if row['Stock?'] == 'Yes':
                mfa_system.StockDict[f"dS_{process_id}"] = msc.Stock(
                    Name=f"dS_{process_id}", P_Res=process_id, Type=1, Indices='t,e', Values=None
                )
                mfa_system.StockDict[f"S_{process_id}"] = msc.Stock(
                    Name=f"S_{process_id}", P_Res=process_id, Type=0, Indices='t,e', Values=None
                )

    mfa_system.Initialize_StockValues()
    print(f"--> Defined {len(mfa_system.ProcessList)} processes and associated stocks.")
    return mfa_system, input_data

In [6]:
# Helper Function 2.4: Define flows and all model parameters (Final Corrected Version)
def define_flows_and_parameters(mfa_system, all_excel_data, dsm_params_config, fomp_params_config):
    """
    Defines all flows and parameters, including the final calculation of
    elemental compositions for all flows.
    """
    # --- Steps 1 & 2 remain the same: Define structure and add material data ---
    flow_definitions = all_excel_data['1_1_Definition_Flows']
    for index, row in flow_definitions.iterrows():
        if pd.notna(row['Name(EN)']):
            import re
            start_id_str = str(row['Process_ID_O'])
            end_id_str = str(row['Process_ID_I'])
            start_match = re.match(r'(\d+)', start_id_str)
            end_match = re.match(r'(\d+)', end_id_str)
            if start_match and end_match:
                start_id, end_id = int(start_match.group(1)), int(end_match.group(1))
                mfa_system.FlowDict[row['Flow_ID']] = msc.Flow(
                    Name=row['Flow_ID'], P_Start=start_id, P_End=end_id, Indices='t,e', Values=None
                )
    mfa_system.Initialize_FlowValues()
    print("--> Defined flows.")
    
    flow_data = all_excel_data['1_2_Data_Flows']
    for flow_id, flow_obj in mfa_system.FlowDict.items():
        if flow_id in flow_data['Flow_ID'].values:
            flow_time_series = flow_data[flow_data['Flow_ID'] == flow_id]
            if len(flow_time_series) == len(mfa_system.IndexTable.Classification['Time'].Items):
                flow_obj.Values[:, 0] = np.array(flow_time_series['Flow_Py']).ravel()
    print("--> Added numerical data to input flows.")

    # --- Step 3: Define all Parameters ---
    parameter_id_counter = 1
    # Dynamic TCs
    dynamic_tc_sheet = all_excel_data.get('2_5_dynamic_tcs')
    if dynamic_tc_sheet is not None:
        dynamic_tcs = create_dynamic_tc_parameters(dynamic_tc_sheet, mfa_system.IndexTable.Classification['Time'].Items)
        for name, values in dynamic_tcs.items():
            mfa_system.ParameterDict[name] = msc.Parameter(Name=name, ID=parameter_id_counter, Values=values, Unit='1')
            parameter_id_counter += 1
    
    # Elemental Contents
    content_definitions = all_excel_data['1_1_Definition_Flows']
    for index, row in content_definitions.iterrows():
        if pd.notna(row['Flow_ID']) and row['Flow_ID'] in mfa_system.FlowDict:
            for element in ['WC', 'DM', 'CC']:
                if element in row and pd.notna(row[element]):
                    mfa_system.ParameterDict[f"{element}_{row['Flow_ID']}"] = msc.Parameter(
                        Name=f"{element}_{row['Flow_ID']}", ID=parameter_id_counter, Values=row[element], Unit='1'
                    )
                    parameter_id_counter += 1
    
    # DSM Parameters
    for process_id, params in dsm_params_config.items():
        for param_name, value in params['lifetimes'].items():
            mfa_system.ParameterDict[f"dsm_{process_id}_{param_name}"] = msc.Parameter(
                Name=f"dsm_{process_id}_{param_name}", ID=parameter_id_counter, P_Res=process_id, Values=value, Unit='yr'
            )
            parameter_id_counter += 1
            
    print(f"--> Defined {len(mfa_system.ParameterDict)} parameters.")

    # --- NEUER SCHRITT 4: Berechne Elementgehalte für primäre Input-Flüsse ---
    print("--> Calculating elemental composition for primary input flows...")
    for flow in mfa_system.FlowDict.values():
        # Check if the material flow for this flow is already filled
        if np.any(flow.Values[:, 0] != 0):
            # Loop through WC, DM, CC and calculate their values
            for i_elem, element_name in enumerate(mfa_system.Elements[1:], 1):
                param_name = f"{element_name}_{flow.Name}"
                if param_name in mfa_system.ParameterDict:
                    content_value = mfa_system.ParameterDict[param_name].Values
                    flow.Values[:, i_elem] = flow.Values[:, 0] * content_value
    
    # --- Final Consistency Check ---
    mfa_system.Consistency_Check()
    print("--> System setup is complete and ready for calculations!")
    
    return mfa_system, all_excel_data

In [7]:
# Helper Function 2.5: Create dynamic TC time series from long table with interpolation
def create_dynamic_tc_parameters(dynamic_tc_data, time_vector):
    """
    Generates time series for TCs by reading a long data table and interpolating
    between the defined data points for each year in the model scope.

    Args:
        dynamic_tc_data (pd.DataFrame): DataFrame loaded from the dynamic TCs sheet.
        time_vector (list): The list of years for the analysis.

    Returns:
        dict: A dictionary of TC names and their complete time series arrays.
    """
    print("--> Generating dynamic TC time series via interpolation...")
    dynamic_tc_dict = {}
    
    # Get a list of all unique TC_IDs from the input sheet
    if 'TC_ID' not in dynamic_tc_data.columns:
        print("--> Warning: 'TC_ID' column not found in dynamic TCs sheet. Skipping.")
        return {}
        
    unique_tc_ids = dynamic_tc_data['TC_ID'].unique()
    
    for tc_id in unique_tc_ids:
        if pd.isna(tc_id):
            continue

        # Filter the dataframe for the current TC_ID
        tc_points = dynamic_tc_data[dynamic_tc_data['TC_ID'] == tc_id]
        
        # Create a pandas Series with the defined data points, using 'Year' as the index
        ts = pd.Series(tc_points['Value'].values, index=tc_points['Year'])
        
        # Create a complete time series by reindexing with all model years
        ts_full = ts.reindex(time_vector)
        
        # Interpolate the missing values (e.g., linear) and fill first/last values
        ts_interpolated = ts_full.interpolate(method='linear', limit_direction='both')
        
        # Store the final numpy array in our dictionary
        dynamic_tc_dict[tc_id] = ts_interpolated.to_numpy()

    print(f"--> Generated {len(dynamic_tc_dict)} dynamic TC parameter(s).")
    return dynamic_tc_dict

In [8]:
# Helper Function 2.6: Calculate TC-based flows (Final Version)
def calculate_tc_flows(mfa_system, max_iterations=10, ignore_outputs_from=None):
    """
    Iteratively calculates TC-based flows. Can ignore outflows from special processes.
    """
    if ignore_outputs_from is None: ignore_outputs_from = []
    
    print(f"--> Calculating TC-based flows (ignoring outputs from: {ignore_outputs_from})")
    for i in range(max_iterations):
        calculated_a_flow = False
        for flow in mfa_system.FlowDict.values():
            if flow.P_Start in ignore_outputs_from: continue

            if np.all(flow.Values[:, 0] == 0):
                param_name = f"TC_{'_'.join(flow.Name.split('_')[1:3])}"
                if param_name in mfa_system.ParameterDict:
                    input_flows = [f for f in mfa_system.FlowDict.values() if f.P_End == flow.P_Start]
                    if all(np.any(f.Values != 0) or f.P_Start == 0 for f in input_flows):
                        sum_input_flows = sum(f.Values[:, 0] for f in input_flows)
                        tc_value = mfa_system.ParameterDict[param_name].Values
                        flow.Values[:, 0] = sum_input_flows * tc_value
                        calculated_a_flow = True
                        for i_elem, element in enumerate(mfa_system.Elements[1:], 1):
                            content_param_name = f"{element}_{flow.Name}"
                            if content_param_name in mfa_system.ParameterDict:
                                content_value = mfa_system.ParameterDict[content_param_name].Values
                                flow.Values[:, i_elem] = flow.Values[:, 0] * content_value
        if not calculated_a_flow:
            print(f"--> System stable after {i+1} iterations.")
            break
    return mfa_system

In [10]:
# Helper Function 2.7: Calculate dynamic stocks (Corrected Return Logic)
def calculate_dynamic_stock(mfa_system, dsm_params_config):
    """
    Calculates dynamic stocks and returns the modified system and a dictionary 
    with detailed results for plotting.
    """
    print("--> Calculating dynamic stocks...")
    time_vector = np.array(mfa_system.IndexTable.Classification['Time'].Items)
    # Initialize the results dictionary at the beginning.
    dsm_details_results = {} 

    # This loop will run for each process defined in your DSM_PARAMS.
    # If DSM_PARAMS is empty or no matching processes are found, the loop is just skipped.
    for process_id, params in dsm_params_config.items():
        # Check if the process ID from the config exists in the current model
        if not any(p.ID == process_id for p in mfa_system.ProcessList):
            print(f"Warning: Process {process_id} from DSM_PARAMS not found in the current model. Skipping.")
            continue

        print(f"    ... running DSM for process {process_id}")
        dsm_details_results[process_id] = {'category_names': [], 'stock_by_category': []}
        
        inflow_flow_name = next((f.Name for f in mfa_system.FlowDict.values() if f.P_End == process_id), None)
        if not inflow_flow_name: continue
        inflow_values = mfa_system.FlowDict[inflow_flow_name].Values
        
        outflow_flow_name = next((f.Name for f in mfa_system.FlowDict.values() if f.P_Start == process_id), None)
        if not outflow_flow_name: continue

        lt_params = params.get('lifetimes', {})
        inflow_split = params.get('inflow_split', [1.0])
        category_names = params.get('category_names', [])
        mean_lifetimes = lt_params.get('Mean', [])
        std_devs = lt_params.get('StdDev', [])
        
        category_labels = [f"{name} ({lt} yrs)" for name, lt in zip(category_names, mean_lifetimes)]
        dsm_details_results[process_id]['category_names'] = category_labels
        
        outflow_total_material = np.zeros(len(time_vector))
        stock_total_material = np.zeros(len(time_vector))
        
        stock_by_category_list = []

        for i in range(len(inflow_split)):
            inflow_category = inflow_values[:, 0] * inflow_split[i]
            if np.sum(inflow_category) > 0:
                dsm_model = dsm.DynamicStockModel(t=time_vector, i=inflow_category, lt={'Type': lt_params.get('Type'), 'Mean': [mean_lifetimes[i]], 'StdDev': [std_devs[i]]})
                s_c = dsm_model.compute_s_c_inflow_driven()
                o_c = dsm_model.compute_o_c_from_s_c()
                if o_c is not None:
                    outflow_total_material += o_c.sum(axis=1)
                    stock_category_ts = s_c.sum(axis=1)
                    stock_total_material += stock_category_ts
                    stock_by_category_list.append(stock_category_ts)
                else:
                    stock_by_category_list.append(np.zeros(len(time_vector)))
            else:
                stock_by_category_list.append(np.zeros(len(time_vector)))
        
        dsm_details_results[process_id]['stock_by_category'] = stock_by_category_list

        # Update system object
        mfa_system.FlowDict[outflow_flow_name].Values[:, 0] = outflow_total_material
        for elem_idx in range(1, inflow_values.shape[1]):
            composition_factor = np.divide(inflow_values[:, elem_idx], inflow_values[:, 0], out=np.zeros_like(inflow_values[:, 0]), where=inflow_values[:, 0]!=0)
            mfa_system.FlowDict[outflow_flow_name].Values[:, elem_idx] = outflow_total_material * composition_factor
            
        dS_values = inflow_values - mfa_system.FlowDict[outflow_flow_name].Values
        mfa_system.StockDict[f'dS_{process_id}'].Values = dS_values
        
        mfa_system.StockDict[f'S_{process_id}'].Values[:, 0] = stock_total_material
        for elem_idx in range(1, inflow_values.shape[1]):
            composition_factor = np.divide(inflow_values[:, elem_idx], inflow_values[:, 0], out=np.zeros_like(inflow_values[:, 0]), where=inflow_values[:, 0]!=0)
            mfa_system.StockDict[f'S_{process_id}'].Values[:, elem_idx] = stock_total_material * composition_factor

    for process_id in dsm_params_config.keys():
        outflow_flow_name = next((f.Name for f in mfa_system.FlowDict.values() if f.P_Start == process_id), None)
        
    print("--> Dynamic stock calculation finished")
    return mfa_system, dsm_details_results
    

In [11]:
# Helper Function 2.7: Calculate final stock changes and balances
def calculate_final_balances(mfa_system):
    """
    Calculates the stock changes (dS) and absolute stocks (S) for all processes.
    This should be called after all flows have been calculated.

    Args:
        mfa_system (odym.MFAsystem): The MFA system object with calculated flows.

    Returns:
        odym.MFAsystem: The MFA system object with calculated stocks.
    """
    print("--> Calculating final stock balances...")
    for process in mfa_system.ProcessList:
        # Check if a stock is associated with this process
        if f"S_{process.ID}" in mfa_system.StockDict:
            
            # Find all input and output flows for this process
            input_flows = [f.Values for f in mfa_system.FlowDict.values() if f.P_End == process.ID]
            output_flows = [f.Values for f in mfa_system.FlowDict.values() if f.P_Start == process.ID]
            
            sum_input_flows = sum(input_flows) if input_flows else 0
            sum_output_flows = sum(output_flows) if output_flows else 0
            
            # Calculate stock change (dS) and absolute stock (S)
            dS_values = sum_input_flows - sum_output_flows
            mfa_system.StockDict[f"dS_{process.ID}"].Values = dS_values
            mfa_system.StockDict[f"S_{process.ID}"].Values = dS_values.cumsum(axis=0)
            
    print("--> Stock balance calculation finished.")
    return mfa_system

### Erklärung: First-Order Model Process (FOMP)

Ein **First-Order Model Process** (Modellprozess erster Ordnung) wird verwendet, um Prozesse zu simulieren, bei denen die Rate des Austrags (z.B. Abbau, Mineralisierung, Emission) direkt von der Menge des im System vorhandenen Materials (dem Lagerbestand) abhängt.

**Analogie:** Man kann es sich wie eine Badewanne mit offenem Abfluss vorstellen. Je mehr Wasser (Lager) in der Wanne ist, desto höher ist der Druck und desto schneller fließt das Wasser ab (Abfluss). Der Abfluss verlangsamt sich, wenn das Lager kleiner wird.

In diesem Modell wird der FOMP verwendet, um die **Mineralisierung von Biokohle im Boden** (Prozess 17, `Lithosphere_Stock`) über die Zeit zu simulieren. Die vereinfachte Formel, die wir verwenden, lautet:

`Mineralisierungsrate(t) = Lager(t-1) * k`

* `Lager(t-1)` ist der Kohlenstoffbestand im Boden aus dem Vorjahr.
* `k` ist die **Abbaurate** oder **Zerfallskonstante**, die angibt, welcher prozentuale Anteil des Lagerbestands pro Jahr abgebaut wird.

Ein kleiner `k`-Wert bedeutet einen langsamen Abbau und eine lange Verweildauer im Boden, was für die Kohlenstoffsequestrierung erwünscht ist.


* Material	Zerfallskonstante k (pro Jahr)	Ungefähre Halbwertszeit	Anmerkung
* Stroh (unbehandelt)	0.1 - 0.5	1.5 - 7 Jahre	Zersetzt sich relativ schnell im Boden.
* Wurzelmasse	0.05 - 0.2	3.5 - 14 Jahre	Etwas stabiler als oberirdisches Stroh.
* Biokohle (labil)	0.01 - 0.05	14 - 70 Jahre	Der Anteil der Biokohle, der sich schneller zersetzt. Dein Wert von 0.032 passt gut in diese Kategorie.
* Biokohle (stabil)	0.0001 - 0.001	700 - 7000 Jahre	Der Großteil der Biokohle ist extrem langlebig und trägt zur langfristigen Kohlenstoffspeicherung bei.


In [12]:
# --- Helper Function 2.8: Calculate First-Order Model Processes (FOMP) (More Robust) ---
def calculate_fomp(mfa_system, fomp_params_config):
    """
    Calculates stock-dependent outflows for all processes defined with FOMP parameters.
    This version includes a check to ensure a stock is defined for the process.
    """
    print("--> Calculating First-Order Model Processes (FOMP)...")
    time_vector = mfa_system.IndexTable.Classification['Time'].Items
    num_years, num_elements = len(time_vector), len(mfa_system.Elements)
    
    for process_id, params in fomp_params_config.items():
        print(f"    ... running FOMP for process {process_id}")
        
        # Get the stock objects first and check if they exist
        stock_s = mfa_system.StockDict.get(f"S_{process_id}")
        stock_ds = mfa_system.StockDict.get(f"dS_{process_id}")
        
        if stock_s is None or stock_ds is None:
            print(f"--> Warning: No stock defined for FOMP process {process_id}. Check the 'Stock?' column in '2_1_Definition_Processes'. Skipping.")
            continue # Skip to the next process in the FOMP loop

        outflow_flow_name = params.get('outflow_id')
        if not outflow_flow_name or outflow_flow_name not in mfa_system.FlowDict:
            print(f"Warning: Outflow '{outflow_flow_name}' for FOMP process {process_id} not found. Skipping.")
            continue
        
        f, k1, k2 = params.get('f', 0), params.get('k1', 0), params.get('k2', 0)
        
        inflows = [flow.Values for flow in mfa_system.FlowDict.values() if flow.P_End == process_id]
        inflow_values = sum(inflows) if inflows else np.zeros((num_years, num_elements))
        
        new_outflow_values = np.zeros_like(inflow_values)
        new_stock_values = np.zeros_like(inflow_values)
        
        for t in range(num_years):
            stock_t_minus_1 = new_stock_values[t-1, :] if t > 0 else 0
            outflow_t = (inflow_values[t, :] * f) + (stock_t_minus_1 * k1) + (inflow_values[t, :] * k2)
            new_outflow_values[t, :] = outflow_t
            stock_t = stock_t_minus_1 + inflow_values[t, :] - outflow_t
            new_stock_values[t, :] = stock_t

        mfa_system.FlowDict[outflow_flow_name].Values = new_outflow_values
        stock_s.Values = new_stock_values
        stock_ds.Values = inflow_values - new_outflow_values
    for process_id, params in fomp_params_config.items():
        outflow_flow_name = params.get('outflow_id')
        
    print("--> FOMP calculation finished." )
    return mfa_system

In [13]:
# --- Helper Function 4.2: Plot Mass Balance Error per Process (New Version) ---
def plot_mass_balance_error(mfa_system_results):
    """
    Creates an interactive bar chart showing the mass balance error for each process.
    Error = Inflows - Outflows - dS. An error of 0 means perfect balance.
    """
    import plotly.graph_objects as go
    from ipywidgets import interact, IntSlider, Dropdown

    process_names = [p.Name for p in mfa_system_results.ProcessList]
    time_items = mfa_system_results.IndexTable.Classification['Time'].Items
    element_items = mfa_system_results.Elements
    
    fig = go.FigureWidget()

    def update_plot(year, element):
        year_index = time_items.index(year)
        element_index = element_items.index(element)
        
        errors = []
        for p in mfa_system_results.ProcessList:
            in_val = sum(f.Values[year_index, element_index] for f in mfa_system_results.FlowDict.values() if f.P_End == p.ID)
            out_val = sum(f.Values[year_index, element_index] for f in mfa_system_results.FlowDict.values() if f.P_Start == p.ID)
            ds_val = mfa_system_results.StockDict.get(f'dS_{p.ID}', None)
            ds_sum = ds_val.Values[year_index, element_index] if ds_val is not None else 0
            
            error = in_val - out_val - ds_sum
            errors.append(error)
        
        # Color bars based on error direction
        colors = ['#d62728' if e > 1e-9 else '#2ca02c' if e < -1e-9 else '#7f7f7f' for e in errors] # Red for positive, Green for negative, Grey for zero

        with fig.batch_update():
            fig.data = [] # Clear previous data
            fig.add_trace(go.Bar(x=process_names, y=errors, marker_color=colors))
            fig.update_layout(
                title=f"Mass Balance Error Check for {element.upper()} in {year}",
                yaxis_title="Error in Mg (positive = mass created)",
                shapes=[dict(type='line', y0=0, y1=0, x0=-0.5, x1=len(process_names)-0.5, line=dict(color='black', width=2))] # Zero line
            )

    # Create widgets
    year_slider = IntSlider(min=time_items[0], max=time_items[-1], step=1, value=time_items[0], description='Year')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    
    interact(update_plot, year=year_slider, element=element_dropdown)
    display(fig)

In [14]:
# Main Calculation Function (Wrapper) - FINAL CORRECTED WORKFLOW
def run_mfa_calculation(mfa_system_configured):
    """
    Orchestrates the full MFA calculation with a correct, sequential workflow.
    """
    import copy
    mfa_system = copy.deepcopy(mfa_system_configured)
    dsm_details = {}
    
    # 1. Define all processes that have special calculation logic
    special_processes = list(DSM_PARAMS.keys()) + list(FOMP_PARAMS.keys())
    
    # 2. Calculate all flows EXCEPT for the outputs of our special processes
    print("\n--- Step 1: Calculating preliminary flows ---")
    mfa_system = calculate_tc_flows(mfa_system, ignore_outputs_from=special_processes)
    
    # 3. Run the special models (DSM and/or FOMP)
    if RUN_DSM_CALCULATION and DSM_PARAMS:
        print("\n--- Step 2: Running Dynamic Stock Model ---")
        mfa_system, dsm_details = calculate_dynamic_stock(mfa_system, DSM_PARAMS)

    if RUN_FOMP_CALCULATION and FOMP_PARAMS:
        print("\n--- Step 3: Running FOMP ---")
        mfa_system = calculate_fomp(mfa_system, FOMP_PARAMS)
    
    # 4. Calculate all remaining flows that depend on the special process outputs
    print("\n--- Step 4: Calculating remaining downstream flows ---")
    mfa_system = calculate_tc_flows(mfa_system) # This second call has no ignore_list
    
    # 5. Calculate final balances for all non-special stocks
    print("\n--- Step 5: Finalizing all stock balances ---")
    mfa_system = calculate_final_balances(mfa_system)
    
    print("\n--> MFA calculation process finished.")
    return mfa_system, dsm_details

# Section 3: Main workflow (execution)

In [15]:

# ====================================================================
# Section 3: Main Execution / Workflow (Corrected Version)
# ====================================================================

# --- 1. Define Model Scope ---
# We call our new function and pass the parameters from the configuration block
model_classification, index_table = define_model_scope(START_YEAR, END_YEAR, ELEMENTS)

# --- 2. Initialize the MFA System Object ---
# The output of the first function is the input for this one
my_mfa_system = initialize_mfa_system(model_classification, index_table)

# --- 3. Load Data and Define Processes/Stocks ---
# The mfa_system object is passed in and gets modified by the function.
# We also receive all excel data to use it in the next steps.
my_mfa_system, all_excel_data = load_and_define_processes(my_mfa_system, EXCEL_FILE_PATH)

# --- 4. Define Flows and Parameters ---
# This function completes the setup of the system object.
# This is the correct call that unpacks the tuple.
my_mfa_system_configured, _ = define_flows_and_parameters(my_mfa_system, all_excel_data, DSM_PARAMS, FOMP_PARAMS)

my_mfa_system_with_results, dsm_details = run_mfa_calculation(my_mfa_system_configured)

# --- 5. Calculation Phase ---
print("\nRunning MFA calculations...")
my_mfa_system_with_results, dsm_details = run_mfa_calculation(my_mfa_system_configured)
print("Calculation complete.")

# --- 6. Display final mass balance (optional) ---
balance = my_mfa_system_with_results.MassBalance()
print("\nFinal Mass Balance Check (Sum of absolute errors per process):")
print(np.abs(balance).sum(axis=0).sum(axis=1))
plot_mass_balance_error(my_mfa_system_with_results)



print("\n✅ System setup is complete and ready for calculations!")

--> Model scope and classifications defined.
--> MFA system object initialized.
--> Excel file '250617_Template_CS0_3.xlsx' loaded successfully.
--> Defined 10 processes and associated stocks.
--> Defined flows.
--> Added numerical data to input flows.
--> Generating dynamic TC time series via interpolation...
--> Generated 12 dynamic TC parameter(s).
--> Defined 57 parameters.
--> Calculating elemental composition for primary input flows...
--> System setup is complete and ready for calculations!

--- Step 1: Calculating preliminary flows ---
--> Calculating TC-based flows (ignoring outputs from: [6, 8])
--> System stable after 2 iterations.

--- Step 2: Running Dynamic Stock Model ---
--> Calculating dynamic stocks...
    ... running DSM for process 6
--> Dynamic stock calculation finished

--- Step 3: Running FOMP ---
--> Calculating First-Order Model Processes (FOMP)...
    ... running FOMP for process 8
--> FOMP calculation finished.

--- Step 4: Calculating remaining downstream f

interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'marker': {'color': [#7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f,
                                   #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f]},
              'type': 'bar',
              'uid': 'da578bf6-795e-4799-8450-1e320f532ccb',
              'x': [Atmosphere, Environment, Cultivation, Harvest, Food, Straw
                    d&C, Use_Straw_Roof, Incineration_Roof, Composting_roof,
                    Roof_EoL],
              'y': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}],
    'layout': {'shapes': [{'line': {'color': 'black', 'width': 2},
                           'type': 'line',
                           'x0': -0.5,
                           'x1': 9.5,
                           'y0': 0,
                           'y1': 0}],
               'template': '...',
               'title': {'text': 'Mass Balance Error Check for MATERIAL in 2025'},
               'yaxis': {'title': {'text': 'Error in Mg (positive = mass created)'}}}
})


✅ System setup is complete and ready for calculations!


### Überprüfung der Massenbilanz

Die folgende Grafik ist das wichtigste Werkzeug zur Überprüfung der Modellkonsistenz. Sie zeigt den Massenbilanzfehler für jeden Prozess, berechnet nach der Formel:

**`Fehler = Σ Zuflüsse - Σ Abflüsse - Lagerveränderung (dS)`**

**Wie man die Grafik liest:**
* **Perfektes Gleichgewicht:** Ein Prozess ist perfekt bilanziert, wenn sein Balken genau auf der Nulllinie liegt.
* **Positiver Fehler (Balken > 0):** Es wurde mehr Masse "erschaffen" als im System sein dürfte. Mögliche Ursache: Ein Abfluss oder eine Lagerbildung fehlt in der Definition.
* **Negativer Fehler (Balken < 0):** Es ist Masse "verschwunden". Mögliche Ursache: Ein Zufluss fehlt oder ein Abfluss wird doppelt gezählt.

Je größer die Abweichung von Null, desto gravierender ist der Fehler in der Modelllogik für diesen Prozess.

# Section 4: Results and visualization

In [16]:
# --- Helper Function 4.1: Create an interactive Sankey plot (Final Corrected Version) ---
def plot_interactive_sankey(mfa_system_results):
    """
    Generates an interactive Sankey diagram with widgets to select the year, 
    element, processes, and a value threshold to hide minor flows.
    """
    import plotly.graph_objects as go
    from ipywidgets import interact, IntSlider, Dropdown, SelectMultiple, FloatSlider

    all_process_names = [p.Name for p in mfa_system_results.ProcessList]
    all_flows = list(mfa_system_results.FlowDict.values())
    time_items = mfa_system_results.IndexTable.Classification['Time'].Items
    element_items = mfa_system_results.Elements
    
    # Determine a reasonable max for the slider, handle case with no flows
    max_flow_value = max(f.Values.max() for f in all_flows) if all_flows and all(f.Values is not None for f in all_flows) else 1

    # IMPORTANT: Create the FigureWidget with an initial, empty Sankey trace
    fig = go.FigureWidget(data=[go.Sankey(node=dict(label=[]), link=dict(source=[], target=[], value=[]))])

    def update_sankey(year, element, processes_to_show, min_flow_value):
        # --- Data Filtering Logic (remains the same) ---
        if not processes_to_show:
            with fig.batch_update():
                fig.data[0].node.label = []
                fig.data[0].link.source = []
                fig.data[0].link.target = []
                fig.data[0].link.value = []
            return

        label_map = {p.ID: i for i, p in enumerate(mfa_system_results.ProcessList) if p.Name in processes_to_show}
        filtered_labels = list(processes_to_show)
        
        year_index = time_items.index(year)
        element_index = element_items.index(element)

        candidate_flows = [f for f in all_flows if f.P_Start in label_map and f.P_End in label_map]
        
        final_flows = [f for f in candidate_flows if f.Values[year_index, element_index] >= min_flow_value]

        # --- THIS IS THE CORRECTED UPDATE LOGIC ---
        # Use batch_update to efficiently update the properties of the EXISTING Sankey trace
        with fig.batch_update():
            if not final_flows:
                # If no flows are left after filtering, show an empty plot but keep the nodes
                fig.data[0].node.label = filtered_labels
                fig.data[0].link.source = []
                fig.data[0].link.target = []
                fig.data[0].link.value = []
            else:
                # Update all properties of the Sankey trace
                fig.data[0].node.label = filtered_labels
                fig.data[0].node.color = "blue"
                fig.data[0].link.source = [label_map[f.P_Start] for f in final_flows]
                fig.data[0].link.target = [label_map[f.P_End] for f in final_flows]
                fig.data[0].link.value = [f.Values[year_index, element_index] for f in final_flows]
            
            # Update layout title
            fig.update_layout(
                title_text=f"MFA Sankey...", 
                font_size=12, 
                height=700,
                margin=dict(l=10, r=10, b=20, t=50) # NEUE ZEILE: Reduziert die Ränder
            )

    # Create widgets
    year_slider = IntSlider(min=time_items[0], max=time_items[-1], step=1, value=time_items[0], description='Year')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element')
    process_selector = SelectMultiple(options=all_process_names, value=list(all_process_names), description='Processes', rows=8)
    threshold_slider = FloatSlider(min=0, max=max_flow_value, step=max_flow_value/100, value=0, description='Min Flow', continuous_update=False, readout_format='.2f')
    
    interact(update_sankey, year=year_slider, element=element_dropdown, processes_to_show=process_selector, min_flow_value=threshold_slider)
    display(fig)

In [17]:
# --- Helper Function 4.3: Plot Inflow, Stock, and Outflow (Final Corrected Version) ---
def plot_process_dynamics(mfa_system_results, process_definitions):
    """
    Creates three side-by-side line charts showing the dynamics of 
    Inflow, Stock, and Outflow, using process type metadata for smarter titles.
    """
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown

    # <<< HIER IST DIE ANPASSUNG: Der korrekte Spaltenname aus deiner Excel-Datei >>>
    PROCESS_TYPE_COLUMN_NAME = 'Process_Type' 

    # Check if the column exists to avoid errors
    has_type_column = PROCESS_TYPE_COLUMN_NAME in process_definitions.columns
    if not has_type_column:
        print(f"Warning: Column '{PROCESS_TYPE_COLUMN_NAME}' not found in '2_1_Definition_Processes'. Smart titles will be disabled.")

    process_options = {p.Name: p.ID for p in mfa_system_results.ProcessList if f"S_{p.ID}" in mfa_system_results.StockDict}
    if not process_options:
        print("No processes with stocks found to plot.")
        return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    
    fig = go.FigureWidget(make_subplots(rows=1, cols=3, subplot_titles=("Inflow", "Stock (S)", "Outflow")))

    def update_plot(process_name, element):
        pid = process_options[process_name]
        element_index = element_items.index(element)

        inflows = [f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_End == pid]
        inflow_ts = sum(inflows) if inflows else np.zeros(len(time_axis))
        stock_ts = mfa_system_results.StockDict[f'S_{pid}'].Values[:, element_index]
        outflows = [f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_Start == pid]
        outflow_ts = sum(outflows) if outflows else np.zeros(len(time_axis))
        
        subplot_titles = (f"Inflow to '{process_name}'", f"Stock in '{process_name}'", f"Outflow from '{process_name}'")
        
        if has_type_column:
            process_type = process_definitions.loc[process_definitions['ID'] == pid, PROCESS_TYPE_COLUMN_NAME].iloc[0]
            if process_type == 'Input':
                subplot_titles = ("Primary System Input", f"Stock in '{process_name}'", f"Outflow from '{process_name}'")
            elif process_type == 'Output':
                subplot_titles = (f"Inflow to '{process_name}'", f"Stock in '{process_name}'", "Final System Output (Sink)")

        with fig.batch_update():
            fig.data, fig.layout.annotations = [], []
            fig.add_trace(go.Scatter(x=time_axis, y=inflow_ts, mode='lines', name='Inflow'), row=1, col=1)
            fig.add_trace(go.Scatter(x=time_axis, y=stock_ts, mode='lines', name='Stock'), row=1, col=2)
            fig.add_trace(go.Scatter(x=time_axis, y=outflow_ts, mode='lines', name='Outflow'), row=1, col=3)
            
            fig.layout.annotations = [
                dict(x=0.155, y=1.05, text=subplot_titles[0], showarrow=False, xref='paper', yref='paper', xanchor='center'),
                dict(x=0.5, y=1.05, text=subplot_titles[1], showarrow=False, xref='paper', yref='paper', xanchor='center'),
                dict(x=0.845, y=1.05, text=subplot_titles[2], showarrow=False, xref='paper', yref='paper', xanchor='center')
            ]
            fig.update_layout(title=f"Dynamics for Process: '{process_name}' | Element: {element.upper()}", height=400, showlegend=False)
            fig.update_xaxes(title_text="Year")
            fig.update_yaxes(title_text="Mass [Mg]", row=1, col=1)

    process_dropdown = Dropdown(options=list(process_options.keys()), description='Process:')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    
    interact(update_plot, process_name=process_dropdown, element=element_dropdown)
    display(fig)

In [18]:
# --- Helper Function 4.4: Plot dynamic stock composition from pre-calculated results ---
def plot_dynamic_stock_composition(dsm_details, mfa_system_results):
    """
    Visualizes the pre-calculated stock composition from the dsm_details dictionary.
    """
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown, Checkbox

    # Create dropdown options from the available results
    process_options = list(dsm_details.keys())
    if not process_options:
        print("DSM calculation was not run or produced no results. Nothing to plot.")
        return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    
    fig = go.FigureWidget()

    def update_plot(process_id, element, show_as_bars):
        details = dsm_details.get(process_id, {})
        stock_by_category = details.get('stock_by_category', [])
        category_labels = details.get('category_names', [])
        
        with fig.batch_update():
            fig.data = [] # Clear previous traces
            chart_type = go.Bar if show_as_bars else go.Scatter
            
            for i, stock_ts in enumerate(stock_by_category):
                # Here you would add logic to get the correct elemental composition if needed
                # For simplicity, we plot the 'material' stock here.
                trace_props = dict(x=time_axis, y=stock_ts, name=category_labels[i], hoverinfo='x+y')
                if not show_as_bars:
                    trace_props.update(mode='lines', line=dict(width=0.5), stackgroup='one')
                fig.add_trace(chart_type(**trace_props))

            process_name = next((p.Name for p in mfa_system_results.ProcessList if p.ID == process_id), "")
            fig.update_layout(
                barmode='stack' if show_as_bars else None,
                title=f"Dynamic Stock Composition for Process: '{process_name}' (Material)",
                xaxis_title="Year", yaxis_title=f"Stock in Mg"
            )

    interact(update_plot, process_id=Dropdown(options=process_options, description='Process:'), 
             element=Dropdown(options=['material'], value='material', description='Element:'), # Simplified for now
             show_as_bars=Checkbox(value=False, description='Show as Bar Chart'))
    display(fig)

In [19]:
# --- Helper Function 4.5: Plot the dynamics of a FOMP process ---
def plot_fomp_dynamics(mfa_system_results, fomp_params_config):
    """
    Creates side-by-side line charts for Inflow, Stock, and Outflow
    for a process calculated with FOMP. Includes interactive widgets.
    """
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown

    # Create a mapping of process names to IDs for the dropdown, only for FOMP processes
    process_options = {
        p.Name: p.ID 
        for p in mfa_system_results.ProcessList 
        if p.ID in fomp_params_config
    }
    if not process_options:
        print("No processes with FOMP parameters are defined in the configuration.")
        return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    
    fig = go.FigureWidget(make_subplots(rows=1, cols=3, subplot_titles=("Total Inflow", "Absolute Stock (S)", "Outflow (Mineralization)")))

    def update_plot(process_name, element):
        pid = process_options[process_name]
        element_index = element_items.index(element)

        # Get the time series data for the selected process
        inflow_ts = sum(f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_End == pid)
        stock_ts = mfa_system_results.StockDict.get(f'S_{pid}').Values[:, element_index]
        outflow_ts = sum(f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_Start == pid)
        
        with fig.batch_update():
            fig.data = [] # Clear existing data
            fig.add_trace(go.Scatter(x=time_axis, y=inflow_ts, mode='lines', name='Inflow'), row=1, col=1)
            fig.add_trace(go.Scatter(x=time_axis, y=stock_ts, mode='lines', name='Stock'), row=1, col=2)
            fig.add_trace(go.Scatter(x=time_axis, y=outflow_ts, mode='lines', name='Outflow'), row=1, col=3)
            
            title_text = f"FOMP Dynamics for Process: '{process_name}' | Element: {element.upper()}"
            fig.update_layout(title_text=title_text, height=400, showlegend=False)
            fig.update_xaxes(title_text="Year")
            fig.update_yaxes(title_text="Mass [Mg]", row=1, col=1)

    # Create widgets for interaction
    process_dropdown = Dropdown(options=list(process_options.keys()), description='Process:')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    
    interact(update_plot, process_name=process_dropdown, element=element_dropdown)
    display(fig)

In [20]:
# --- Helper Function 4.6: Plot the dynamics of selected flows over time ---
def plot_flow_dynamics(mfa_system_results):
    """
    Creates an interactive line chart to show the development of selected
    flows over time for a chosen element.
    """
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown, SelectMultiple

    # Create options for the widgets
    flow_options = sorted(list(mfa_system_results.FlowDict.keys()))
    if not flow_options:
        print("No flows found in the system to plot.")
        return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    
    # Use FigureWidget for efficient updates
    fig = go.FigureWidget()

    def update_plot(flows_to_show, element):
        # Clear previous data
        fig.data = []
        
        if not flows_to_show:
            # Optional: Add a message if nothing is selected
            fig.update_layout(title_text=f"Please select one or more flows to display.")
            return

        element_index = element_items.index(element)
        
        # Add a trace for each selected flow
        for flow_id in flows_to_show:
            flow_obj = mfa_system_results.FlowDict.get(flow_id)
            if flow_obj:
                time_series = flow_obj.Values[:, element_index]
                fig.add_trace(go.Scatter(x=time_axis, y=time_series, mode='lines', name=flow_id))
        
        # Update layout and title
        fig.update_layout(
            title=f"Time Series for Selected Flows ({element.upper()})",
            xaxis_title="Year",
            yaxis_title="Mass in Mg",
            hovermode="x unified" # Shows all values for a given year on hover
        )

    # Create widgets
    flow_selector = SelectMultiple(options=flow_options, value=[flow_options[0]], description='Flows:', rows=10)
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    
    interact(update_plot, flows_to_show=flow_selector, element=element_dropdown)
    display(fig)

In [21]:
# ====================================================================
# Section 4: Results & Visualization (Corrected)
# ====================================================================

# --- 4.1 Interactive Sankey Diagram ---
print("Displaying interactive Sankey diagram:")
plot_interactive_sankey(my_mfa_system_with_results)

# --- 4.2. Display final mass balance (optional) ---
balance = my_mfa_system_with_results.MassBalance()
print("\nFinal Mass Balance Check (Sum of absolute errors per process):")
print(np.abs(balance).sum(axis=0).sum(axis=1))
plot_mass_balance_error(my_mfa_system_with_results)

# --- 4.3 Process Dynamics Plot ---
print("\nDisplaying Inflow-Stock-Outflow Dynamics:")
# This is the single, correct call to the function, providing both required arguments.
plot_process_dynamics(my_mfa_system_with_results, all_excel_data['2_1_Definition_Processes'])

# --- 4.4 Dynamic Stock Composition Plot ---
print("\nDisplaying Dynamic Stock Composition:")
# Check if the DSM was run and details were generated before plotting
if dsm_details:
    plot_dynamic_stock_composition(dsm_details, my_mfa_system_with_results)
else:
    print("--> DSM calculation was skipped. No stock composition to display.")

# --- 4.5 FOMP Dynamics Plot ---
print("\nDisplaying FOMP Dynamics (once calculated):")
# Check if the FOMP was run and params exist before plotting
if FOMP_PARAMS:
    plot_fomp_dynamics(my_mfa_system_with_results, FOMP_PARAMS)
else:
    print("--> No FOMP parameters defined. Nothing to display.")





Displaying interactive Sankey diagram:


interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'link': {'source': [0, 1, 2, 3, 3, 4, 4, 5, 6, 7, 7, 8, 9, 9],
                       'target': [2, 2, 3, 4, 5, 0, 1, 6, 9, 0, 1, 0, 8, 7],
                       'value': [100.0, 100.0, 200.0, 100.0, 100.0, 50.0, 50.0,
                                 100.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
              'node': {'color': 'blue',
                       'label': [Atmosphere, Environment, Cultivation, Harvest,
                                 Food, Straw d&C, Use_Straw_Roof,
                                 Incineration_Roof, Composting_roof, Roof_EoL]},
              'type': 'sankey',
              'uid': '0c561588-8a38-41da-bc3e-f51044da9ff8'}],
    'layout': {'font': {'size': 12},
               'height': 700,
               'margin': {'b': 20, 'l': 10, 'r': 10, 't': 50},
               'template': '...',
               'title': {'text': 'MFA Sankey...'}}
})

Prüfe Stock-Änderung: dS_0
  Indices String: 't,e'
  Values Shape: (26, 4)
Prüfe Stock-Änderung: dS_1
  Indices String: 't,e'
  Values Shape: (26, 4)
Prüfe Stock-Änderung: dS_6
  Indices String: 't,e'
  Values Shape: (26, 4)
Prüfe Stock-Änderung: dS_7
  Indices String: 't,e'
  Values Shape: (26, 4)
Prüfe Stock-Änderung: dS_8
  Indices String: 't,e'
  Values Shape: (26, 4)

Final Mass Balance Check (Sum of absolute errors per process):
[8.12150347e-13 2.41584530e-13 0.00000000e+00 4.26325641e-13
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 3.55271368e-14]


interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

...

interactive(children=(Dropdown(description='Process:', options=(6,), value=6), Dropdown(description='Element:'…

FigureWidget({
    'data': [{'hoverinfo': 'x+y',
              'line': {'width': 0.5},
              'mode': 'lines',
              'name': 'Insulation Material (20 yrs)',
              'stackgroup': 'one',
              'type': 'scatter',
              'uid': '9d937717-c954-45ca-af76-d2329567a889',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'y': array([  80.        ,  169.76      ,  269.6       ,  379.84      ,
                           500.8       ,  632.8       ,  776.16      ,  931.2       ,
                          1098.24      , 1277.6       , 1469.6       , 1477.6       ,
                          1618.4       , 1728.8       , 1805.59999992, 1845.59997698,
                          1912.15744047, 2007.08913662, 2130.69562882, 2275.52722778,
                          2425.73504954, 2550.096799


Displaying FOMP Dynamics (once calculated):


interactive(children=(Dropdown(description='Process:', options=('Composting_roof',), value='Composting_roof'),…

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'Inflow',
              'type': 'scatter',
              'uid': '4f2c094b-5803-441f-828a-73682f08ddd9',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'xaxis': 'x',
              'y': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
                          1.14660629e-06, 2.00000014e+00, 4.24399900e+00, 4.74000001e+00,
                          5.25212669e+00, 5.87101599e+00, 8.33511804e+00, 1.10484820e+01,
                          1.22006183e+01, 1.33047300e+01, 1.44407301e+01, 1.56087364e+01,
                          1.18857464e+01, 1.12240538e+01, 1.50661308e+01, 1.85117785e+01,
                          2.86977482e+01, 3.74159160e+01, 4.52709183e+01, 5.36343024e+01,
                          5.80449923e+01, 6

# Section 5: Export Results

In [22]:
# ====================================================================
# Section 5: Export Results
# ====================================================================

def export_results_to_excel(mfa_system_results, output_filename="mfa_results.xlsx"):
    """
    Exports all calculated flows and stocks into a single Excel file with multiple sheets.
    """
    print(f"\n--> Exporting results to '{output_filename}'...")
    
    time_index = mfa_system_results.IndexTable.Classification['Time'].Items
    elements = mfa_system_results.Elements
    
    with pd.ExcelWriter(output_filename) as writer:
        # --- Export Flows ---
        flow_data_rows = []
        for name, flow_obj in mfa_system_results.FlowDict.items():
            for i, year in enumerate(time_index):
                row = {'Flow_ID': name, 'Year': year}
                for j, element in enumerate(elements):
                    row[element] = flow_obj.Values[i, j]
                flow_data_rows.append(row)
        df_flows = pd.DataFrame(flow_data_rows)
        df_flows.to_excel(writer, sheet_name='Flows_ts', index=False)
        
        # --- Export Stocks ---
        stock_data_rows = []
        for name, stock_obj in mfa_system_results.StockDict.items():
            for i, year in enumerate(time_index):
                row = {'Stock_ID': name, 'Year': year}
                for j, element in enumerate(elements):
                    row[element] = stock_obj.Values[i, j]
                stock_data_rows.append(row)
        df_stocks = pd.DataFrame(stock_data_rows)
        df_stocks.to_excel(writer, sheet_name='Stocks_ts', index=False)
        
    print("--> Export complete.")

In [23]:
# ====================================================================
# Section 5: Export Results
# ====================================================================

# Call the export function
export_results_to_excel(my_mfa_system_with_results, output_filename="rye_mfa_results_v1.xlsx")


--> Exporting results to 'rye_mfa_results_v1.xlsx'...
--> Export complete.


<img src="system_flow_diagram.svg" alt="system_flow_diagram">

## 0 Load packages

This cell imports all necessary packages. It also loads the ODYM framework and the bioDYM_addon, both are included as files in the project (see folder /framework). 

## 3 MFA Calculations

Now, the solution of the MFA is calculated. Since most flows have either input data or TCs and substance contents are given, they can be easily calculated. However, this system includes a dynamic stock modeling (dsm) and a first order model process (FOMP) for the mineralization of carbon in soil. The idea is that first, all flows are calculated with TCs that are independent of dsm or FOMP. Then, dsm is performed and subsequently, all flows up to the FOMP are calculated. After that, the bioDYM_addon functions are used to calculate the mineralization. Finally, all following flows and stocks are calculated.

### 3.1 Solution MFA

### 3.1.1 Solution MFA pt. I (until MBC dynamic stock modelling)


### 3.1.4 Solution MFA pt. IV (FOMP mineralization process)

The carbon mineralization process in the soil is calculated with a first order decay model according to (Cayuela et al., 2010) based on (Robertson & Paul, 2000): 

$$ C_{remaining} (t)=f \cdot exp⁡(-k_{1} \cdot t)+(100\%-f) \cdot exp⁡(-k_{2} \cdot t) $$

To keep calculations simple, it is assumed that this is the only equation that leads to outflows of the process, the remaining fractions of the material accumulate as stock without any emissions. (Cayuela et al., 2010) give parameter values for green waste biochar, they are used here.

\
\
Literature

Cayuela, M. L., Oenema, O., Kuikman, P. J., Bakker, R. R., & Van GROENIGEN, J. W. (2010). Bioenergy by-products as soil amendments? Implications for carbon sequestration and greenhouse gas emissions: C AND N DYNAMICS FROM BIOENERGY BY-PRODUCTS IN SOIL. GCB Bioenergy, no-no. https://doi.org/10.1111/j.1757-1707.2010.01055.x

Robertson, G. P., & Paul, E. (2000). Decomposition and Soil Organic Matter Dynamics. Decomposition and Soil Organic Matter Dynamics., 104–116. https://doi.org/10.1007/978-1-4612-1224-9_8